# Credit Card Fraud Detection

End-to-end fraud detection on the ULB Credit Card Fraud dataset (Kaggle). Covers EDA, preprocessing, model training (Logistic Regression, Random Forest), evaluation, and key findings.


## SECTION 1 — Imports & Data Loading


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download the dataset (cached locally after first run)
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Path to dataset files:", path)


In [ ]:
df = pd.read_csv(path + "/creditcard.csv")

print(df.shape)
print(df.head())
print(df.info())
print(df.describe())
print(df['Class'].value_counts())


## SECTION 2 — Class Distribution

**What:** bar chart of raw counts + pie chart of percentage split between legitimate and fraud.

**Why:** this 0.17% fraud rate is the reason SMOTE is needed later, and why accuracy alone would be a misleading metric for this dataset.


In [ ]:
class_counts = df['Class'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart — raw counts
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, f"{v:,}", ha='center', va='bottom')

# Pie chart — percentage split
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%',
            colors=['steelblue', 'crimson'], explode=[0, 0.3])
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig('../images/class_imbalance.png', dpi=150)
plt.show()

print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")
print(f"Legit: {class_counts[0]:,} | Fraud: {class_counts[1]:,}")


## SECTION 3–5 — Amount Analysis, Zero-Amount Transactions & Time Analysis


In [ ]:
# --- Amount: Fraud vs Legitimate ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0], hue='Class',
            palette=['steelblue', 'crimson'], legend=False)
axes[0].set_title('Transaction Amount by Class')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Legitimate', 'Fraud'])
axes[0].set_ylabel('Amount (\u20ac)')

df[df['Class']==0]['Amount'].plot(kind='hist', bins=50, alpha=0.6, ax=axes[1],
                                   color='steelblue', label='Legitimate', density=True)
df[df['Class']==1]['Amount'].plot(kind='hist', bins=50, alpha=0.6, ax=axes[1],
                                   color='crimson', label='Fraud', density=True)
axes[1].set_title('Amount Distribution: Fraud vs Legitimate')
axes[1].set_xlabel('Amount (\u20ac)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/amount_distribution.png', dpi=150)
plt.show()

print("=== Amount Statistics by Class ===")
print(df.groupby('Class')['Amount'].describe().round(2))

# --- Time analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class']==0]['Time'].plot(kind='hist', bins=48, alpha=0.6, ax=axes[0],
                                 color='steelblue', label='Legitimate', density=True)
df[df['Class']==1]['Time'].plot(kind='hist', bins=48, alpha=0.6, ax=axes[0],
                                 color='crimson', label='Fraud', density=True)
axes[0].set_title('Transaction Time Distribution')
axes[0].set_xlabel('Time (seconds)')
axes[0].legend()

# Note: this 'Hour' is absolute hour across the full 2-day dataset (0-47),
# not hour-of-day. It is kept as a model feature later (consistent with the
# original results), so do not rename or drop it.
df['Hour'] = (df['Time'] / 3600).astype(int)
fraud_by_hour = df[df['Class']==1].groupby('Hour').size()
fraud_by_hour.plot(kind='bar', ax=axes[1], color='crimson', edgecolor='black')
axes[1].set_title('Fraud Count by Hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Number of Frauds')

plt.tight_layout()
plt.savefig('../images/time_analysis.png', dpi=150)
plt.show()

# --- Zero-amount transactions ---
zero_amount = df[df['Amount'] == 0]
print(f"\n=== Zero Amount Transactions ===")
print(f"Total: {len(zero_amount)}")
print(f"Fraud in zero-amount: {zero_amount['Class'].sum()}")
print(zero_amount['Class'].value_counts())


## SECTION 6 — Feature Separation


In [ ]:
# Which features separate fraud from legitimate best?
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

v_features = [f'V{i}' for i in range(1, 29)]
mean_diff = abs(fraud[v_features].mean() - legit[v_features].mean())
mean_diff_sorted = mean_diff.sort_values(ascending=False)

print("=== Top 10 Features That Differ Most (Fraud vs Legitimate) ===")
print(mean_diff_sorted.head(10).round(4))

plt.figure(figsize=(12, 5))
mean_diff_sorted.head(10).plot(kind='bar', color='crimson', edgecolor='black')
plt.title('Top 10 Features: Mean Difference Between Fraud and Legitimate')
plt.xlabel('Feature')
plt.ylabel('Absolute Mean Difference')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../images/vfeature_importance_eda.png', dpi=150)
plt.show()

top3 = mean_diff_sorted.head(3).index.tolist()
print(f"\nTop 3 most separating features: {top3}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, feat in enumerate(top3):
    legit[feat].plot(kind='hist', bins=50, alpha=0.6, ax=axes[i],
                      color='steelblue', label='Legitimate', density=True)
    fraud[feat].plot(kind='hist', bins=50, alpha=0.6, ax=axes[i],
                      color='crimson', label='Fraud', density=True)
    axes[i].set_title(f'{feat} Distribution')
    axes[i].legend()
    axes[i].set_xlabel(feat)

plt.tight_layout()
plt.savefig('../images/top_features_distribution.png', dpi=150)
plt.show()


## SECTION 6B — Advanced EDA (Phase 1: Correlation, Hourly Trends, Outliers)

Inserted here deliberately — *before* preprocessing/scaling, so these analyses use the original, human-readable `Amount`/`Time` values rather than scaled ones.


### 6B.1 — Correlation Heatmap (Balanced Subsample)

**What:** shows how strongly every pair of features moves together, as colors.

**Why:** on the full imbalanced data, correlation with fraud gets drowned out — balancing 492 fraud + 492 legit reveals real relationships.


In [ ]:
fraud_df = df[df['Class'] == 1]
legit_df = df[df['Class'] == 0]
legit_sample = legit_df.sample(n=len(fraud_df), random_state=42)
balanced_df = pd.concat([fraud_df, legit_sample], axis=0).reset_index(drop=True)

# drop the 'Hour' helper column here — it's a derived feature, not useful in a correlation view
corr_matrix = balanced_df.drop(columns=['Hour']).corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False, linewidths=0.5)
plt.title('Correlation Heatmap (Balanced Subsample: 492 Fraud + 492 Legit)', fontsize=14)
plt.tight_layout()
plt.savefig('../images/correlation_heatmap_balanced.png', dpi=150)
plt.show()

class_corr = corr_matrix['Class'].drop('Class').sort_values(key=abs, ascending=False)
print("Top 10 features correlated with Class (fraud):")
print(class_corr.head(10))


### 6B.2 — Hourly Trend Line Plots

**What:** line charts of transaction count / total amount / mean amount, by hour-of-day.

**Why:** trends reveal timing patterns bar charts hide — e.g. fraud spikes overnight.

**Note:** this uses `.assign()` to add a temporary `HourOfDay` column (0–23, cyclical) *without* writing it back onto `df` — keeping it separate from the `Hour` column (0–47, absolute) created in Section 5, which is intentionally kept as a model feature.


In [ ]:
hourly_view = df.assign(HourOfDay=(df['Time'] // 3600) % 24)

hourly_stats = hourly_view.groupby(['HourOfDay', 'Class']).agg(
    transaction_count=('Amount', 'count'),
    total_amount=('Amount', 'sum'),
    mean_amount=('Amount', 'mean')
).reset_index()

legit_hourly = hourly_stats[hourly_stats['Class'] == 0]
fraud_hourly = hourly_stats[hourly_stats['Class'] == 1]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(legit_hourly['HourOfDay'], legit_hourly['transaction_count'], label='Legit', color='steelblue')
axes[0].plot(fraud_hourly['HourOfDay'], fraud_hourly['transaction_count'], label='Fraud', color='crimson')
axes[0].set_title('Transaction Count by Hour of Day'); axes[0].set_xlabel('Hour'); axes[0].legend()

axes[1].plot(legit_hourly['HourOfDay'], legit_hourly['total_amount'], label='Legit', color='steelblue')
axes[1].plot(fraud_hourly['HourOfDay'], fraud_hourly['total_amount'], label='Fraud', color='crimson')
axes[1].set_title('Total Amount by Hour of Day'); axes[1].set_xlabel('Hour'); axes[1].legend()

axes[2].plot(legit_hourly['HourOfDay'], legit_hourly['mean_amount'], label='Legit', color='steelblue')
axes[2].plot(fraud_hourly['HourOfDay'], fraud_hourly['mean_amount'], label='Fraud', color='crimson')
axes[2].set_title('Mean Amount by Hour of Day'); axes[2].set_xlabel('Hour'); axes[2].legend()

plt.tight_layout()
plt.savefig('../images/hourly_trends.png', dpi=150)
plt.show()


### 6B.3 — IQR Outlier Check on V14

**What:** flags values outside Q1−1.5×IQR to Q3+1.5×IQR as statistical outliers.

**Why:** check whether V14's extreme values are fraud signal *before* ever considering removing them — don't delete the thing you're trying to detect.


In [ ]:
Q1 = df['V14'].quantile(0.25)
Q3 = df['V14'].quantile(0.75)
IQR = Q3 - Q1
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df['V14'] < lower_bound) | (df['V14'] > upper_bound)]
print(f"V14 IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Total V14 outliers: {len(outliers)}")
print(f"Fraud rate within V14 outliers: {outliers['Class'].mean()*100:.2f}%")
print(f"Fraud rate in full dataset: {df['Class'].mean()*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Class', y='V14', data=df, ax=axes[0], hue='Class',
            palette=['steelblue', 'crimson'], legend=False)
axes[0].set_title('V14 \u2014 Before Outlier Removal')

df_no_outliers = df[(df['V14'] >= lower_bound) & (df['V14'] <= upper_bound)]
sns.boxplot(x='Class', y='V14', data=df_no_outliers, ax=axes[1], hue='Class',
            palette=['steelblue', 'crimson'], legend=False)
axes[1].set_title('V14 \u2014 After Outlier Removal (comparison only, not applied)')

plt.tight_layout()
plt.savefig('../images/v14_outlier_analysis.png', dpi=150)
plt.show()


## SECTION 7–9 — Preprocessing, Train/Test Split & SMOTE


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
                              confusion_matrix,
                              roc_auc_score,
                              precision_score,
                              recall_score,
                              f1_score,
                              RocCurveDisplay,
                              PrecisionRecallDisplay)

# --- Preprocessing ---
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])
df_model = df.drop(['Amount', 'Time'], axis=1)

X = df_model.drop('Class', axis=1)
y = df_model['Class']

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

# --- Train/Test Split ---
# stratify=y ensures both splits maintain the 0.17% fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"Fraud in train: {y_train.sum()}")
print(f"Fraud in test:  {y_test.sum()}")

# --- Handle Imbalance with SMOTE (training data only) ---
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"Train size: {X_train_res.shape[0]}")
print(f"Class distribution:\n", pd.Series(y_train_res).value_counts())


## SECTION 10 — Model Training: Logistic Regression & Random Forest

**What:** train two classifiers on the SMOTE-balanced training data.

**Why:** compare a simple linear model against an ensemble model, since they behave very differently on imbalanced classification problems.


In [ ]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_res, y_train_res)

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print("=== Logistic Regression \u2014 Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=['Legitimate', 'Fraud']))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_res, y_train_res)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\n=== Random Forest \u2014 Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['Legitimate', 'Fraud']))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))


## SECTION 11 — Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Confusion Matrix \u2014 Logistic Regression')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
axes[1].set_title('Confusion Matrix \u2014 Random Forest')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../images/confusion_matrices.png', dpi=150)
plt.show()

print(f"LR \u2014 False Alarms (FP): {cm_lr[0,1]} | Missed Frauds (FN): {cm_lr[1,0]}")
print(f"RF \u2014 False Alarms (FP): {cm_rf[0,1]} | Missed Frauds (FN): {cm_rf[1,0]}")


## SECTION 12–13 — ROC Curves & Precision-Recall Curves


In [ ]:
# --- ROC Curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_prob_lr, name='Logistic Regression', ax=axes[0])
axes[0].set_title('ROC Curve \u2014 Logistic Regression')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
axes[0].legend()

RocCurveDisplay.from_predictions(y_test, y_prob_rf, name='Random Forest', ax=axes[1])
axes[1].set_title('ROC Curve \u2014 Random Forest')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/roc_curves.png', dpi=150)
plt.show()

# --- Precision-Recall Curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

PrecisionRecallDisplay.from_predictions(y_test, y_prob_lr, name='Logistic Regression', ax=axes[0])
axes[0].set_title('Precision-Recall \u2014 Logistic Regression')

PrecisionRecallDisplay.from_predictions(y_test, y_prob_rf, name='Random Forest', ax=axes[1])
axes[1].set_title('Precision-Recall \u2014 Random Forest')

plt.tight_layout()
plt.savefig('../images/precision_recall_curves.png', dpi=150)
plt.show()


## SECTION 14 — Model Comparison Summary

Computed dynamically from the actual results above, rather than hardcoded — so this table always matches whatever the models just produced, even if you re-run with different hyperparameters later.


In [ ]:
metrics_table = pd.DataFrame({
    'Logistic Regression': [
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_prob_lr),
        cm_lr[0, 1],
        cm_lr[1, 0]
    ],
    'Random Forest': [
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_score(y_test, y_prob_rf),
        cm_rf[0, 1],
        cm_rf[1, 0]
    ]
}, index=['Fraud Precision', 'Fraud Recall', 'Fraud F1-Score', 'ROC-AUC', 'False Alarms (FP)', 'Missed Frauds (FN)'])

print("=" * 58)
print("SECTION 14: MODEL COMPARISON SUMMARY")
print("=" * 58)
print(metrics_table.round(4))


## SECTION 15 — Key Insights


In [ ]:
print("\n" + "=" * 58)
print("SECTION 15: KEY INSIGHTS")
print("=" * 58)

insights = """
INSIGHT 1 \u2014 Card Testing Behavior
Fraudulent transactions have a median amount of \u20ac9.25 vs \u20ac22.00
for legitimate ones. The 25th percentile of fraud is just \u20ac1.00,
meaning a quarter of all fraud transactions are \u20ac1 or less.
This is consistent with known 'card testing' behavior where
fraudsters verify stolen cards with micro-transactions before
attempting larger ones.

INSIGHT 2 \u2014 Zero-Amount Fraud Rate is 9x Higher
Zero-amount transactions (authorization checks) have a fraud
rate of 1.48% vs 0.17% overall. These should be flagged as
elevated risk events in real monitoring systems.

INSIGHT 3 \u2014 Feature Separation
Features V3, V14, V17 show the strongest separation between
fraud and legitimate transactions. Legitimate transactions
cluster tightly near 0, while fraud spreads into extreme
negative values \u2014 making these reliable model signals.

INSIGHT 4 \u2014 The Precision-Recall Tradeoff
Logistic Regression catches the large majority of fraud but
generates far more false alarms. Random Forest cuts false
alarms dramatically but misses more fraud cases. This is a
real business decision: high recall protects revenue, high
precision protects customer experience.

INSIGHT 5 \u2014 Why Accuracy is Misleading
A model predicting everything as legitimate achieves 99.83%
accuracy but catches zero fraud. This is why Precision,
Recall, F1, and ROC-AUC are the correct metrics for
imbalanced datasets like this one.

INSIGHT 6 \u2014 V14 Outliers Carry Signal, Not Noise
Outlier rows in V14 (by IQR) have a fraud rate far above the
dataset average \u2014 confirming these extreme values are part
of the fraud signal. They were intentionally kept in the
modeling data rather than removed as "noise."
"""
print(insights)

print("\n\u2705 Project complete. All charts saved as PNG files.")
print("Next step: Push to GitHub with a README.md")
